# Customer Churn Prediction

End-to-end workflow: data cleaning → exploratory data analysis → feature engineering → model training → evaluation.

In [ ]:
# ===== Core libraries =====
import pandas as pd                # data loading & manipulation
import numpy as np                 # numerical operations

# ===== Visualization libraries =====
import matplotlib.pyplot as plt    # base plotting engine
import seaborn as sns              # statistical plots built on matplotlib

# ===== Machine learning utilities =====
from sklearn.model_selection import train_test_split          # split data into train/test sets
from sklearn.preprocessing import LabelEncoder, StandardScaler  # encode categories / scale numeric features
from sklearn.metrics import (
    accuracy_score,       # overall % of correct predictions
    recall_score,         # % of actual churners correctly identified
    roc_auc_score,        # ability to rank churners above non-churners
    confusion_matrix,     # breakdown of TP/FP/TN/FN
    classification_report,  # precision/recall/f1 summary per class
    roc_curve              # points to plot the ROC curve
)

# Suppress non-critical warnings (e.g. sklearn deprecation notices) to keep the output clean
import warnings
warnings.filterwarnings("ignore")


## Global Plot Styling

Set a consistent, modern look for every chart in this notebook (theme, color palette, fonts) so we don't have to repeat styling code in every cell.

In [ ]:
# ===== Global chart styling (applies to every plot below) =====

# Use seaborn's "whitegrid" theme for a clean, modern background with light gridlines
sns.set_theme(
    style="whitegrid",
    palette="viridis",       # perceptually-uniform, colorblind-friendly color palette
    font_scale=1.05           # slightly larger text for readability
)

# Custom color palette used for two-category comparisons (e.g. Churn vs No Churn)
CHURN_PALETTE = ["#2EC4B6", "#E71D36"]   # teal = retained, red = churned

# Fine-tune matplotlib defaults so every figure looks polished by default
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.labelweight": "medium",
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

def style_axis(ax, title=None, xlabel=None, ylabel=None):
    """Small helper to keep chart styling consistent: adds a bold title,
    removes the top/right border ('spines'), and labels the axes."""
    if title:
        ax.set_title(title, pad=14)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    sns.despine(ax=ax)   # remove the top and right chart borders for a cleaner look
    return ax


In [ ]:
# Load the raw Telco customer churn dataset from Excel
# NOTE: update this path to wherever the file lives on your machine
df = pd.read_excel(r"C:\Users\popo\Desktop\Telco_customer_churn.xlsx")

# Preview the first 5 rows to sanity-check the data loaded correctly
df.head()


In [ ]:
# Quick shape check: how many customers (rows) and features (columns) do we have?
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


In [ ]:
# Show column names, data types, and non-null counts for every column
df.info()


In [ ]:
# List all column names as a plain Python list (handy for copy/pasting elsewhere)
df.columns.tolist()


In [ ]:
# Summary statistics for every column (numeric + categorical thanks to include="all"),
# transposed (.T) so each row is a column of the original dataframe
df.describe(include="all").T


In [ ]:
# Count of unique values per column, sorted ascending
# Useful to spot binary/categorical columns (low unique count) vs. IDs (high unique count)
df.nunique().sort_values()


In [ ]:
# ===== Missing value check =====
missing = df.isnull().sum().sort_values(ascending=False)

missing_df = pd.DataFrame({
    "Missing Values": missing,
    "Missing %": (missing / len(df)) * 100
})

# Only display columns that actually have missing values
missing_df[missing_df["Missing Values"] > 0]


In [ ]:
# Check for fully duplicated rows across the entire dataset
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
# Check for duplicate Customer IDs specifically (each customer should appear once)
print("Duplicate Customer IDs:", df["CustomerID"].duplicated().sum())


In [ ]:
# ===== Standardize column names =====
# lowercase, strip whitespace, replace spaces/hyphens with underscores
# -> makes columns easier to reference in code (e.g. df.tenure_months)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns.tolist()


In [ ]:
# Confirm the renamed columns look correct
df.head()


In [ ]:
# "total_charges" was likely read in as text (object) rather than a number.
# Convert to numeric; any value that can't be converted becomes NaN (errors="coerce")
df["total_charges"] = pd.to_numeric(
    df["total_charges"],
    errors="coerce"
)


In [ ]:
# How many rows failed to convert to a number?
df["total_charges"].isnull().sum()


In [ ]:
# Inspect the rows where total_charges is missing (usually brand-new customers
# with 0 tenure, so total_charges was blank in the source file)
df[df["total_charges"].isnull()][
    ["customerid", "tenure_months", "monthly_charges", "total_charges"]
]

# Fill those missing values with 0 (no charges accrued yet)
df["total_charges"] = df["total_charges"].fillna(0)


In [ ]:
# Confirm total_charges is now fully numeric with no missing values
df["total_charges"]


In [ ]:
# Raw counts of churned (1) vs retained (0) customers
df["churn_value"].value_counts()


In [ ]:
# Same thing but as a percentage of the total customer base
df["churn_value"].value_counts(normalize=True) * 100


In [ ]:
# ===== Chart: overall churn distribution =====
fig, ax = plt.subplots(figsize=(6, 4.5))

counts = df["churn_value"].value_counts().sort_index()
bars = ax.bar(
    ["No Churn", "Churn"],
    counts.values,
    color=CHURN_PALETTE,      # teal = retained, red = churned
    edgecolor="black",
    linewidth=0.8,
    width=0.6
)

# Add the exact count + percentage on top of each bar for extra clarity
total = counts.sum()
for bar, value in zip(bars, counts.values):
    pct = value / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.01,
        f"{value:,}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=10, fontweight="bold"
    )

style_axis(ax, title="Customer Churn Distribution", xlabel="Churn", ylabel="Number of Customers")
ax.margins(y=0.15)
plt.tight_layout()
plt.show()


In [ ]:
# Reusable helper: for a given categorical column, compute the count, number
# of churners, and churn rate (%) within each category, sorted by churn rate
def churn_analysis(column):

    result = (
        df.groupby(column)["churn_value"]
        .agg(["count", "sum", "mean"])
        .reset_index()
    )

    result["churn_rate_%"] = result["mean"] * 100

    return result.sort_values(
        "churn_rate_%",
        ascending=False
    )


In [ ]:
# Churn rate broken down by gender
churn_analysis("gender")


In [ ]:
# ===== Chart: churn rate by gender =====
fig, ax = plt.subplots(figsize=(7, 5))

sns.barplot(
    data=df,
    x="gender",
    y="churn_value",
    hue="gender",
    palette="viridis",
    edgecolor="black",
    linewidth=0.8,
    legend=False,
    ax=ax
)

# Label each bar with its churn rate as a percentage
for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: f"{v*100:.1f}%", padding=3, fontsize=9, fontweight="bold")

style_axis(ax, title="Churn Rate by Gender", xlabel="Gender", ylabel="Churn Rate")
ax.set_ylim(0, max(df.groupby("gender")["churn_value"].mean()) * 1.25)
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate broken down by senior citizen status
churn_analysis("senior_citizen")


In [ ]:
# ===== Chart: churn rate by senior citizen =====
fig, ax = plt.subplots(figsize=(7, 5))

sns.barplot(
    data=df,
    x="senior_citizen",
    y="churn_value",
    hue="senior_citizen",
    palette="viridis",
    edgecolor="black",
    linewidth=0.8,
    legend=False,
    ax=ax
)

# Label each bar with its churn rate as a percentage
for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: f"{v*100:.1f}%", padding=3, fontsize=9, fontweight="bold")

style_axis(ax, title="Churn Rate by Senior Citizen Status", xlabel="Senior Citizen", ylabel="Churn Rate")
ax.set_ylim(0, max(df.groupby("senior_citizen")["churn_value"].mean()) * 1.25)
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate broken down by whether the customer has a partner
churn_analysis("partner")


In [ ]:
# ===== Chart: churn rate by partner =====
fig, ax = plt.subplots(figsize=(7, 5))

sns.barplot(
    data=df,
    x="partner",
    y="churn_value",
    hue="partner",
    palette="viridis",
    edgecolor="black",
    linewidth=0.8,
    legend=False,
    ax=ax
)

# Label each bar with its churn rate as a percentage
for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: f"{v*100:.1f}%", padding=3, fontsize=9, fontweight="bold")

style_axis(ax, title="Churn Rate by Partner Status", xlabel="Partner", ylabel="Churn Rate")
ax.set_ylim(0, max(df.groupby("partner")["churn_value"].mean()) * 1.25)
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate broken down by whether the customer has dependents
churn_analysis("dependents")


In [ ]:
# ===== Chart: churn rate by dependents =====
fig, ax = plt.subplots(figsize=(7, 5))

sns.barplot(
    data=df,
    x="dependents",
    y="churn_value",
    hue="dependents",
    palette="viridis",
    edgecolor="black",
    linewidth=0.8,
    legend=False,
    ax=ax
)

# Label each bar with its churn rate as a percentage
for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: f"{v*100:.1f}%", padding=3, fontsize=9, fontweight="bold")

style_axis(ax, title="Churn Rate by Dependents", xlabel="Dependents", ylabel="Churn Rate")
ax.set_ylim(0, max(df.groupby("dependents")["churn_value"].mean()) * 1.25)
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate broken down by contract type (month-to-month, one year, two year)
churn_analysis("contract")


In [ ]:
# ===== Chart: churn rate by contract =====
fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=df,
    x="contract",
    y="churn_value",
    hue="contract",
    palette="viridis",
    edgecolor="black",
    linewidth=0.8,
    legend=False,
    ax=ax
)

# Label each bar with its churn rate as a percentage
for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: f"{v*100:.1f}%", padding=3, fontsize=9, fontweight="bold")

style_axis(ax, title="Churn Rate by Contract Type", xlabel="Contract", ylabel="Churn Rate")
ax.set_ylim(0, max(df.groupby("contract")["churn_value"].mean()) * 1.25)
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate broken down by internet service type
churn_analysis("internet_service")


In [ ]:
# ===== Chart: churn rate by internet service =====
fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=df,
    x="internet_service",
    y="churn_value",
    hue="internet_service",
    palette="viridis",
    edgecolor="black",
    linewidth=0.8,
    legend=False,
    ax=ax
)

# Label each bar with its churn rate as a percentage
for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: f"{v*100:.1f}%", padding=3, fontsize=9, fontweight="bold")

style_axis(ax, title="Churn Rate by Internet Service", xlabel="Internet Service", ylabel="Churn Rate")
ax.set_ylim(0, max(df.groupby("internet_service")["churn_value"].mean()) * 1.25)
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate broken down by payment method
churn_analysis("payment_method")


In [ ]:
# ===== Chart: churn rate by payment method =====
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=df,
    x="payment_method",
    y="churn_value",
    hue="payment_method",
    palette="viridis",
    edgecolor="black",
    linewidth=0.8,
    legend=False,
    ax=ax
)

# Label each bar with its churn rate as a percentage
for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: f"{v*100:.1f}%", padding=3, fontsize=9, fontweight="bold")

style_axis(ax, title="Churn Rate by Payment Method", xlabel="Payment Method", ylabel="Churn Rate")
ax.set_ylim(0, max(df.groupby("payment_method")["churn_value"].mean()) * 1.25)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate broken down by whether the customer has tech support
churn_analysis("tech_support")


In [ ]:
# ===== Chart: churn rate by tech support =====
fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=df,
    x="tech_support",
    y="churn_value",
    hue="tech_support",
    palette="viridis",
    edgecolor="black",
    linewidth=0.8,
    legend=False,
    ax=ax
)

# Label each bar with its churn rate as a percentage
for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: f"{v*100:.1f}%", padding=3, fontsize=9, fontweight="bold")

style_axis(ax, title="Churn Rate by Tech Support", xlabel="Tech Support", ylabel="Churn Rate")
ax.set_ylim(0, max(df.groupby("tech_support")["churn_value"].mean()) * 1.25)
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate broken down by whether the customer has online security
churn_analysis("online_security")


In [ ]:
# ===== Chart: churn rate by online security =====
fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=df,
    x="online_security",
    y="churn_value",
    hue="online_security",
    palette="viridis",
    edgecolor="black",
    linewidth=0.8,
    legend=False,
    ax=ax
)

# Label each bar with its churn rate as a percentage
for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: f"{v*100:.1f}%", padding=3, fontsize=9, fontweight="bold")

style_axis(ax, title="Churn Rate by Online Security", xlabel="Online Security", ylabel="Churn Rate")
ax.set_ylim(0, max(df.groupby("online_security")["churn_value"].mean()) * 1.25)
plt.tight_layout()
plt.show()


In [ ]:
# ===== Chart: distribution of customer tenure (in months) =====
fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(
    data=df,
    x="tenure_months",
    bins=30,
    kde=True,                     # overlay a smooth density curve
    color="#3A86FF",
    edgecolor="white",
    linewidth=0.5,
    alpha=0.85,
    ax=ax
)
# Style the KDE line separately so it stands out from the bars
if ax.lines:
    ax.lines[-1].set_color("#FF006E")
    ax.lines[-1].set_linewidth(2.5)

style_axis(ax, title="Customer Tenure Distribution", xlabel="Tenure (Months)", ylabel="Number of Customers")
plt.tight_layout()
plt.show()


In [ ]:
# Bucket the continuous tenure_months column into readable groups for easier comparison
bins = [0, 6, 12, 24, 48, 72]

labels = [
    "0-6 Months",
    "7-12 Months",
    "13-24 Months",
    "25-48 Months",
    "49-72 Months"
]

df["tenure_group"] = pd.cut(
    df["tenure_months"],
    bins=bins,
    labels=labels,
    include_lowest=True
)


In [ ]:
# Churn rate broken down by tenure group
churn_analysis("tenure_group")


In [ ]:
# ===== Chart: churn rate by tenure group =====
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=df,
    x="tenure_group",
    y="churn_value",
    hue="tenure_group",
    palette="mako",
    edgecolor="black",
    linewidth=0.8,
    legend=False,
    ax=ax
)

for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: f"{v*100:.1f}%", padding=3, fontsize=9, fontweight="bold")

style_axis(ax, title="Churn Rate by Customer Tenure", xlabel="Tenure Group", ylabel="Churn Rate")
ax.set_ylim(0, df.groupby("tenure_group", observed=True)["churn_value"].mean().max() * 1.25)
plt.tight_layout()
plt.show()


In [ ]:
# ===== Chart: monthly charges distribution, churned vs retained =====
fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="churn_value",
    y="monthly_charges",
    hue="churn_value",
    palette=CHURN_PALETTE,
    legend=False,
    width=0.5,
    linewidth=1.2,
    fliersize=3,
    ax=ax
)
ax.set_xticks([0, 1])
ax.set_xticklabels(["No Churn", "Churn"])

style_axis(ax, title="Monthly Charges vs Churn", xlabel="Churn", ylabel="Monthly Charges ($)")
plt.tight_layout()
plt.show()


In [ ]:
# ===== Chart: total charges distribution, churned vs retained =====
fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="churn_value",
    y="total_charges",
    hue="churn_value",
    palette=CHURN_PALETTE,
    legend=False,
    width=0.5,
    linewidth=1.2,
    fliersize=3,
    ax=ax
)
ax.set_xticks([0, 1])
ax.set_xticklabels(["No Churn", "Churn"])

style_axis(ax, title="Total Charges vs Churn", xlabel="Churn", ylabel="Total Charges ($)")
plt.tight_layout()
plt.show()


In [ ]:
# ===== Chart: tenure vs monthly charges, colored by churn =====
fig, ax = plt.subplots(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="tenure_months",
    y="monthly_charges",
    hue="churn_value",
    palette=CHURN_PALETTE,
    alpha=0.6,
    s=35,
    edgecolor="white",
    linewidth=0.3,
    ax=ax
)

style_axis(ax, title="Tenure vs Monthly Charges", xlabel="Tenure (Months)", ylabel="Monthly Charges ($)")
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ["No Churn", "Churn"], title="Churn", frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix between the key numeric features and churn
numeric_cols = [
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "churn_value"
]

correlation = df[numeric_cols].corr()

correlation


In [ ]:
# ===== Chart: correlation heatmap =====
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    correlation,
    annot=True,             # print the correlation values on each cell
    cmap="coolwarm",        # blue = negative, red = positive correlation
    fmt=".2f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Correlation"},
    vmin=-1, vmax=1,
    ax=ax
)

style_axis(ax, title="Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# List of categorical columns we'll use to explore churn drivers
categorical_columns = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "payment_method"
]


In [ ]:
# Print a churn breakdown table for every categorical column, one after another
for col in categorical_columns:

    print("\n" + "=" * 60)
    print(col.upper())
    print("=" * 60)

    print(churn_analysis(col))


In [ ]:
# Build a single combined table of churn rate per category, across ALL categorical
# columns at once, so we can rank the biggest churn "drivers"
churn_driver_results = []

for col in categorical_columns:

    temp = (
        df.groupby(col)["churn_value"]
        .mean()
        .reset_index()
    )

    temp["churn_rate_%"] = temp["churn_value"] * 100
    temp["variable"] = col

    churn_driver_results.append(temp)

churn_drivers = pd.concat(
    churn_driver_results,
    ignore_index=True
)

churn_drivers = churn_drivers.sort_values(
    "churn_rate_%",
    ascending=False
)

# Top 20 category values with the highest churn rate across the whole dataset
churn_drivers.head(20)


In [ ]:
# ===== Feature engineering: prepare data for modeling =====

# Drop identifier / leakage columns that shouldn't be used to train the model
# (e.g. churn_label/churn_reason/churn_score are derived from churn itself)
drop_columns = [
    "customer_id",
    "count",
    "country",
    "state",
    "churn_label",
    "churn_reason",
    "churn_score"
]

model_df = df.drop(
    columns=drop_columns,
    errors="ignore"   # ignore columns that don't exist (e.g. after renaming)
)


In [ ]:
# Also drop the tenure_group column we created earlier for EDA purposes only
# (tenure_months, the raw numeric version, is still in the data for modeling)
model_df = model_df.drop(
    columns=["tenure_group"],
    errors="ignore"
)


In [ ]:
# Split into features (X) and target (y)
X = model_df.drop(
    columns=["churn_value"]
)

y = model_df["churn_value"]


In [ ]:
# One-hot encode all categorical/text columns into numeric 0/1 columns
# drop_first=True avoids redundant columns (prevents multicollinearity)
X = pd.get_dummies(
    X,
    drop_first=True
)


In [ ]:
# Preview the fully numeric feature matrix used for modeling
X.head()


In [ ]:
# Split into training (80%) and testing (20%) sets
# stratify=y keeps the churn/no-churn ratio consistent in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [ ]:
# Confirm the resulting shapes of the train/test splits
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


In [ ]:
# Standardize features (mean=0, std=1) - required for distance/gradient-based
# models like Logistic Regression to perform well and converge properly
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # fit on train, then transform train
X_test_scaled = scaler.transform(X_test)         # transform test using the SAME scaler (no leakage)


In [ ]:
# ===== Model 1: Logistic Regression (baseline linear model) =====
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,     # allow enough iterations to converge
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_train
)


In [ ]:
# Generate predictions and churn probabilities on the held-out test set
logistic_pred = logistic_model.predict(
    X_test_scaled
)

logistic_prob = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]   # probability of the positive class (churn = 1)


In [ ]:
# Evaluate Logistic Regression performance
logistic_accuracy = accuracy_score(
    y_test,
    logistic_pred
)

logistic_recall = recall_score(
    y_test,
    logistic_pred
)

logistic_roc_auc = roc_auc_score(
    y_test,
    logistic_prob
)

print("Logistic Regression")
print("--------------------")
print("Accuracy:", logistic_accuracy)
print("Recall:", logistic_recall)
print("ROC-AUC:", logistic_roc_auc)


In [ ]:
# ===== Chart: Logistic Regression confusion matrix =====
cm = confusion_matrix(
    y_test,
    logistic_pred
)

fig, ax = plt.subplots(figsize=(6.5, 5.5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Count"},
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"],
    annot_kws={"fontsize": 13, "fontweight": "bold"},
    ax=ax
)

style_axis(ax, title="Logistic Regression - Confusion Matrix", xlabel="Predicted", ylabel="Actual")
plt.tight_layout()
plt.show()


In [ ]:
# Full precision / recall / F1-score breakdown per class
print(
    classification_report(
        y_test,
        logistic_pred
    )
)


In [ ]:
# ===== Model 2: Random Forest (ensemble of decision trees) =====
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,           # number of trees in the forest
    random_state=42,
    class_weight="balanced"     # compensate for imbalanced churn/no-churn classes
)

rf_model.fit(
    X_train,       # tree-based models don't require feature scaling
    y_train
)


In [ ]:
# Generate predictions and churn probabilities using the Random Forest model
rf_pred = rf_model.predict(X_test)

rf_prob = rf_model.predict_proba(
    X_test
)[:, 1]


In [ ]:
# Evaluate Random Forest performance
rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

rf_recall = recall_score(
    y_test,
    rf_pred
)

rf_roc_auc = roc_auc_score(
    y_test,
    rf_prob
)

print("Random Forest")
print("--------------------")
print("Accuracy:", rf_accuracy)
print("Recall:", rf_recall)
print("ROC-AUC:", rf_roc_auc)


In [ ]:
# Combine both models' metrics into one comparison table
model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "Accuracy": [
        logistic_accuracy,
        rf_accuracy
    ],

    "Recall": [
        logistic_recall,
        rf_recall
    ],

    "ROC-AUC": [
        logistic_roc_auc,
        rf_roc_auc
    ]
})

model_results


In [ ]:
# ===== Chart: ROC curve comparison between models =====
logistic_fpr, logistic_tpr, _ = roc_curve(
    y_test,
    logistic_prob
)

rf_fpr, rf_tpr, _ = roc_curve(
    y_test,
    rf_prob
)

fig, ax = plt.subplots(figsize=(8, 6.5))

ax.plot(
    logistic_fpr,
    logistic_tpr,
    color="#3A86FF",
    linewidth=2.5,
    label=f"Logistic Regression (AUC = {logistic_roc_auc:.3f})"
)

ax.plot(
    rf_fpr,
    rf_tpr,
    color="#FF006E",
    linewidth=2.5,
    label=f"Random Forest (AUC = {rf_roc_auc:.3f})"
)

# Diagonal reference line = performance of random guessing (AUC = 0.5)
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    linewidth=1.5,
    label="Random Guess (AUC = 0.500)"
)

ax.fill_between(logistic_fpr, logistic_tpr, alpha=0.05, color="#3A86FF")
ax.fill_between(rf_fpr, rf_tpr, alpha=0.05, color="#FF006E")

style_axis(ax, title="ROC Curve Comparison", xlabel="False Positive Rate", ylabel="True Positive Rate")
ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
# Rank features by how much they contributed to the Random Forest's decisions
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(20)


In [ ]:
# ===== Chart: top 15 most important features driving churn predictions =====
top_features = feature_importance.head(15)

fig, ax = plt.subplots(figsize=(10, 7.5))

bars = sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature",
    hue="Feature",
    palette="rocket",
    edgecolor="black",
    linewidth=0.7,
    legend=False,
    ax=ax
)

# Add the exact importance score at the end of each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=4, fontsize=9, fontweight="bold")

style_axis(ax, title="Top 15 Churn Prediction Features", xlabel="Importance", ylabel="Feature")
ax.set_xlim(0, top_features["Importance"].max() * 1.15)
plt.tight_layout()
plt.show()


In [ ]:
# Install XGBoost (gradient-boosted trees) if it isn't already available
!pip install xgboost


In [ ]:
# ===== Model 3: XGBoost (gradient boosting) =====
from xgboost import XGBClassifier


In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,        # number of boosting rounds
    learning_rate=0.05,      # step size shrinkage to prevent overfitting
    max_depth=4,             # max depth of each tree
    subsample=0.8,           # fraction of rows sampled per tree
    colsample_bytree=0.8,    # fraction of features sampled per tree
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)


In [ ]:
# Generate predictions and churn probabilities using XGBoost
xgb_pred = xgb_model.predict(X_test)

xgb_prob = xgb_model.predict_proba(
    X_test
)[:, 1]


In [ ]:
# Evaluate XGBoost performance
xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)

xgb_recall = recall_score(
    y_test,
    xgb_pred
)

xgb_roc_auc = roc_auc_score(
    y_test,
    xgb_prob
)

print("XGBoost")
print("--------------------")
print("Accuracy:", xgb_accuracy)
print("Recall:", xgb_recall)
print("ROC-AUC:", xgb_roc_auc)


In [ ]:
# ===== Bonus chart: compare all three models side-by-side =====
all_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Accuracy": [logistic_accuracy, rf_accuracy, xgb_accuracy],
    "Recall": [logistic_recall, rf_recall, xgb_recall],
    "ROC-AUC": [logistic_roc_auc, rf_roc_auc, xgb_roc_auc],
})

melted = all_results.melt(id_vars="Model", var_name="Metric", value_name="Score")

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=melted,
    x="Metric",
    y="Score",
    hue="Model",
    palette=["#3A86FF", "#FFBE0B", "#FF006E"],
    edgecolor="black",
    linewidth=0.7,
    ax=ax
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3, fontsize=8, fontweight="bold")

style_axis(ax, title="Model Performance Comparison", xlabel="Metric", ylabel="Score")
ax.set_ylim(0, 1.1)
ax.legend(title="Model", frameon=True, loc="lower right")
plt.tight_layout()
plt.show()
